# 6. Squat Rep Counting (Google Colab)

This notebook starts after step 5 has produced processed squat feature files.

Input:
- `squat_feature_index.csv`
- `squat_features/*.npy`

Output:
- predicted rep counts per video
- a prototype FSM-based squat counter
- basic evaluation against labeled squat counts

## 1. Paths

In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')
ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data' / 'LLSP' / 'annotation_cleaned'

SQUAT_FEATURE_INDEX_CSV = ANNOTATION_DIR / 'squat_feature_index.csv'
RUN_SUMMARY_CSV = ANNOTATION_DIR / 'squat_feature_summary.csv'
REP_COUNT_RESULTS_CSV = ANNOTATION_DIR / 'squat_rep_count_results.csv'

print('ANNOTATION_DIR =', ANNOTATION_DIR)
print('SQUAT_FEATURE_INDEX_CSV =', SQUAT_FEATURE_INDEX_CSV)
print('RUN_SUMMARY_CSV =', RUN_SUMMARY_CSV)
print('REP_COUNT_RESULTS_CSV =', REP_COUNT_RESULTS_CSV)

## 2. Imports

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 3. Load Feature Index

In [ ]:
feature_index = pd.read_csv(SQUAT_FEATURE_INDEX_CSV)
feature_index.head()

In [ ]:
print('rows =', len(feature_index))
print('missing feature files =', int((~feature_index['feature_path'].map(lambda p: Path(p).exists())).sum()))

## 4. Feature Schema

These columns match the output order from step 5.

In [ ]:
FEATURE_COLUMNS = [
    'frame_idx',
    'left_knee_angle',
    'right_knee_angle',
    'avg_knee_angle',
    'knee_flex',
    'left_hip_angle',
    'right_hip_angle',
    'avg_hip_angle',
    'hip_center_y',
    'knee_center_y',
    'ankle_center_y',
    'hip_drop',
    'leg_extension',
    'hip_velocity',
    'frame_valid',
    'mean_conf',
]

len(FEATURE_COLUMNS)

## 5. Load One Feature File

In [ ]:
def load_feature_frame(path: Path) -> pd.DataFrame:
    arr = np.load(path)
    if arr.ndim != 2 or arr.shape[1] != len(FEATURE_COLUMNS):
        raise ValueError(f'Expected [T, {len(FEATURE_COLUMNS)}], got {arr.shape} for {path}')
    return pd.DataFrame(arr, columns=FEATURE_COLUMNS)


sample_row = feature_index.iloc[0]
sample_df = load_feature_frame(Path(sample_row['feature_path']))
print('video =', sample_row['name'])
print('shape =', sample_df.shape)
sample_df.head()

## 6. Counter Parameters

In [ ]:
FSM_CFG = {
    'min_conf': 0.25,
    'min_valid_ratio': 0.5,
    'enter_down': 20.0,
    'enter_bottom': 55.0,
    'exit_bottom': 40.0,
    'back_to_up': 15.0,
    'min_bottom_frames': 2,
}

FSM_CFG

## 7. Squat FSM

In [ ]:
@dataclass
class CountResult:
    pred_count: int
    state_trace: list[str]
    event_frames: list[int]


def count_squat_reps(features: pd.DataFrame, cfg: dict) -> CountResult:
    count = 0
    state = 'UP'
    state_trace = []
    event_frames = []
    bottom_frames = 0

    for i, row in features.iterrows():
        valid_frame = (row['frame_valid'] >= cfg['min_valid_ratio']) and (row['mean_conf'] >= cfg['min_conf'])
        knee_flex = float(row['knee_flex'])

        if not valid_frame:
            state_trace.append(state)
            continue

        if state == 'UP':
            bottom_frames = 0
            if knee_flex > cfg['enter_down']:
                state = 'DESCENDING'

        elif state == 'DESCENDING':
            if knee_flex > cfg['enter_bottom']:
                state = 'BOTTOM'
                bottom_frames = 1
            elif knee_flex < cfg['back_to_up']:
                state = 'UP'

        elif state == 'BOTTOM':
            if knee_flex > cfg['exit_bottom']:
                bottom_frames += 1
            else:
                if bottom_frames >= cfg['min_bottom_frames']:
                    state = 'ASCENDING'
                else:
                    state = 'DESCENDING'

        elif state == 'ASCENDING':
            if knee_flex < cfg['back_to_up']:
                count += 1
                event_frames.append(int(row['frame_idx']))
                state = 'UP'
                bottom_frames = 0
            elif knee_flex > cfg['enter_bottom']:
                state = 'BOTTOM'
                bottom_frames = 1

        state_trace.append(state)

    return CountResult(pred_count=count, state_trace=state_trace, event_frames=event_frames)

## 8. Inspect One Count Trace

In [ ]:
sample_result = count_squat_reps(sample_df, FSM_CFG)
true_count = float(sample_row['count'])

print('video =', sample_row['name'])
print('true_count =', true_count)
print('pred_count =', sample_result.pred_count)
print('event_frames =', sample_result.event_frames[:20])

In [ ]:
plot_df = sample_df.copy()
plot_df['state'] = sample_result.state_trace

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(plot_df['knee_flex'], label='knee_flex')
axes[0].axhline(FSM_CFG['enter_down'], linestyle='--', label='enter_down')
axes[0].axhline(FSM_CFG['enter_bottom'], linestyle='--', label='enter_bottom')
axes[0].axhline(FSM_CFG['back_to_up'], linestyle='--', label='back_to_up')
for frame in sample_result.event_frames:
    axes[0].axvline(frame, color='green', alpha=0.3)
axes[0].legend()
axes[0].set_ylabel('knee_flex')

axes[1].plot(plot_df['hip_drop'], label='hip_drop')
axes[1].plot(plot_df['hip_velocity'], label='hip_velocity')
axes[1].legend()
axes[1].set_ylabel('motion signal')

state_to_num = {'UP': 0, 'DESCENDING': 1, 'BOTTOM': 2, 'ASCENDING': 3}
axes[2].plot(plot_df['state'].map(state_to_num), label='state')
axes[2].plot(plot_df['frame_valid'], label='frame_valid')
axes[2].legend()
axes[2].set_ylabel('state / valid')
axes[2].set_xlabel('frame')

plt.tight_layout()
plt.show()

## 9. Batch Counting

In [ ]:
results = []

for i, row in feature_index.iterrows():
    features = load_feature_frame(Path(row['feature_path']))
    count_result = count_squat_reps(features, FSM_CFG)
    true_count = float(row['count']) if pd.notna(row['count']) else np.nan
    pred_count = float(count_result.pred_count)
    abs_error = abs(pred_count - true_count) if pd.notna(true_count) else np.nan

    results.append({
        'name': row['name'],
        'split': row.get('split', ''),
        'feature_path': row['feature_path'],
        'true_count': true_count,
        'pred_count': pred_count,
        'abs_error': abs_error,
        'events_found': len(count_result.event_frames),
        'frames_total': len(features),
        'frames_valid': int(features['frame_valid'].sum()),
        'mean_conf': float(features['mean_conf'].mean()),
    })

    if (i + 1) % 25 == 0 or (i + 1) == len(feature_index):
        print(f'[{i + 1}/{len(feature_index)}] counted')

results_df = pd.DataFrame(results)
results_df.to_csv(REP_COUNT_RESULTS_CSV, index=False)
print('saved =', REP_COUNT_RESULTS_CSV)

## 10. Evaluation

In [ ]:
results_df.head()

In [ ]:
print('MAE overall =', results_df['abs_error'].mean())
print('RMSE overall =', np.sqrt(np.mean((results_df['pred_count'] - results_df['true_count']) ** 2)))
print('Within-1 accuracy =', np.mean(results_df['abs_error'] <= 1.0))

In [ ]:
results_df.groupby('split')[['abs_error']].mean()

In [ ]:
worst = results_df.sort_values('abs_error', ascending=False).head(15)
worst[['name', 'split', 'true_count', 'pred_count', 'abs_error', 'mean_conf', 'frames_valid']]

## 11. Next Tuning Directions

If the count errors are still high, tune:

- `enter_down`
- `enter_bottom`
- `exit_bottom`
- `back_to_up`
- `min_bottom_frames`

You can also try different signals:
- `knee_flex`
- `hip_drop`
- combinations of `knee_flex` and `hip_velocity`